# 02 - Spark Processing

This notebook uses PySpark to process and analyze the Twitter US Airline Sentiment dataset.

## Objectives
- Load the tweet dataset into Spark
- Clean and prepare tweet text
- Analyze sentiment distribution
- Compare sentiment across airlines
- Identify major complaint reasons
- Extract common words from negative tweets
- Find highly retweeted tweets

In [2]:
import os
os.environ["JAVA_HOME"] = r"C:\Users\hp\AppData\Local\Programs\Eclipse Adoptium\jdk-17.0.18.8-hotspot"
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PYSPARK_PYTHON"] = r"C:\Users\hp\anaconda3\envs\pyspark_env\python.exe"

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lower, regexp_replace, trim

spark = SparkSession.builder \
    .appName("02 - Spark Processing") \
    .config("spark.driver.memory", "2g") \
    .config("spark.ui.enabled", "false") \
    .getOrCreate()

print(" Spark Ready!", spark.version)

 Spark Ready! 4.1.1


In [ ]:
import os
os.environ["JAVA_HOME"] = r"C:\Users\hp\AppData\Local\Programs\Eclipse Adoptium\jdk-17.0.18.8-hotspot"
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PYSPARK_PYTHON"] = r"C:\Users\hp\anaconda3\envs\pyspark_env\python.exe"

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lower, regexp_replace, trim

spark = SparkSession.builder \
    .appName("02 - Spark Processing") \
    .config("spark.driver.memory", "2g") \
    .config("spark.ui.enabled", "false") \
    .getOrCreate()

print(" Spark Ready!", spark.version)

## Load Dataset

The dataset is loaded from the project data folder using PySpark.  
Spark allows us to process the tweet data efficiently and prepares it for large-scale analysis.

## Data Cleaning and Preparation

We select relevant columns and remove rows with missing values.
We also create a cleaned version of tweet text.

## Text Cleaning

This step removes URLs, mentions, and special characters,
and converts text to lowercase for better analysis.

In [3]:
df = spark.read.csv(
    r"C:\Users\hp\BIGDATAPROJECT\data\Tweets.csv",
    header=True, inferSchema=True
)

df_clean = df.select(
    "tweet_id","airline_sentiment","airline_sentiment_confidence",
    "negativereason","airline","retweet_count",
    "text","tweet_created","tweet_location"
).dropna(subset=["tweet_id","airline_sentiment","airline","text"])

df_clean = df_clean.withColumn("clean_text",
    trim(lower(regexp_replace(regexp_replace(
        regexp_replace(col("text"), r"http\S+",""),
    r"@\w+",""),r"[^a-zA-Z\s]","")))
)

print(" Data loaded! Rows:", df_clean.count())

 Data loaded! Rows: 14632


## Overall Sentiment Distribution

This section counts how many tweets are positive, neutral, and negative.

In [4]:
from pyspark.sql.functions import count, desc, avg

print("=== Overall Sentiment Distribution ===")
df_clean.groupBy("airline_sentiment") \
    .count() \
    .orderBy(desc("count")) \
    .show()

=== Overall Sentiment Distribution ===
+-----------------+-----+
|airline_sentiment|count|
+-----------------+-----+
|         negative| 9170|
|          neutral| 3099|
|         positive| 2363|
+-----------------+-----+



## Sentiment by Airline

This shows how customer sentiment differs across airlines.

In [5]:
print("=== Sentiment by Airline ===")
df_clean.groupBy("airline", "airline_sentiment") \
    .count() \
    .orderBy("airline", desc("count")) \
    .show(30)

=== Sentiment by Airline ===
+--------------+-----------------+-----+
|       airline|airline_sentiment|count|
+--------------+-----------------+-----+
|      American|         negative| 1958|
|      American|          neutral|  463|
|      American|         positive|  336|
|         Delta|         negative|  953|
|         Delta|          neutral|  723|
|         Delta|         positive|  544|
|     Southwest|         negative| 1185|
|     Southwest|          neutral|  664|
|     Southwest|         positive|  570|
|    US Airways|         negative| 2263|
|    US Airways|          neutral|  381|
|    US Airways|         positive|  269|
|        United|         negative| 2630|
|        United|          neutral|  697|
|        United|         positive|  492|
|Virgin America|         negative|  181|
|Virgin America|          neutral|  171|
|Virgin America|         positive|  152|
+--------------+-----------------+-----+



## Average Sentiment Confidence

This calculates the average confidence score for each airline's sentiment classification.

In [6]:
print("=== Average Sentiment Confidence per Airline ===")
df_clean.groupBy("airline") \
    .agg(avg("airline_sentiment_confidence").alias("avg_confidence")) \
    .orderBy(desc("avg_confidence")) \
    .show()

=== Average Sentiment Confidence per Airline ===
+--------------+------------------+
|       airline|    avg_confidence|
+--------------+------------------+
|    US Airways|0.9215784414692754|
|      American|0.9172919840406241|
|        United|0.9008202409007601|
|     Southwest|0.8864989251756928|
|Virgin America|0.8760861111111107|
|         Delta|0.8698515315315324|
+--------------+------------------+



## Top Negative Reasons

This identifies the most common reasons for negative customer feedback.

In [7]:
print("=== Top Negative Reasons ===")
df_clean.filter(
    (col("airline_sentiment") == "negative") &
    col("negativereason").isNotNull()
).groupBy("negativereason") \
 .count() \
 .orderBy(desc("count")) \
 .show()

=== Top Negative Reasons ===
+--------------------+-----+
|      negativereason|count|
+--------------------+-----+
|Customer Service ...| 2908|
|         Late Flight| 1663|
|          Can't Tell| 1189|
|    Cancelled Flight|  846|
|        Lost Luggage|  722|
|          Bad Flight|  580|
|Flight Booking Pr...|  529|
|Flight Attendant ...|  481|
|           longlines|  178|
|     Damaged Luggage|   74|
+--------------------+-----+



## Frequent Words in Negative Tweets

This extracts common words from negative tweets to understand complaint patterns.

In [8]:
from pyspark.sql.functions import explode, split

stopwords = ["the","to","a","i","and","of","for","in","is","it",
             "my","on","was","this","so","have","are","with","me","you",
             "at","but","be","we","our","us","im","just","get","its"]

print("=== Top 20 Words in Negative Tweets ===")
df_clean.filter(col("airline_sentiment") == "negative") \
    .select(explode(split(col("clean_text"), " ")).alias("word")) \
    .filter(col("word") != "") \
    .filter(~col("word").isin(stopwords)) \
    .groupBy("word") \
    .count() \
    .orderBy(desc("count")) \
    .show(20)

=== Top 20 Words in Negative Tweets ===
+---------+-----+
|     word|count|
+---------+-----+
|   flight| 2904|
|     your| 1335|
|      not| 1323|
|       no| 1281|
|     that| 1113|
|cancelled|  914|
|      now|  817|
|       an|  775|
|     been|  768|
|     from|  735|
|  service|  735|
|    hours|  645|
|      can|  620|
|     help|  605|
|     hold|  605|
| customer|  598|
|     time|  581|
|       do|  565|
|       up|  552|
|     they|  537|
+---------+-----+
only showing top 20 rows


## Most Retweeted Tweets

This shows tweets that gained the most attention and engagement.

In [9]:
print("=== Top 10 Most Retweeted Tweets ===")
df_clean.filter(col("retweet_count") > 0) \
    .orderBy(desc("retweet_count")) \
    .select("airline","airline_sentiment","retweet_count","text") \
    .show(10, truncate=70)

=== Top 10 Most Retweeted Tweets ===
+----------+-----------------+-------------+----------------------------------------------------------------------+
|   airline|airline_sentiment|retweet_count|                                                                  text|
+----------+-----------------+-------------+----------------------------------------------------------------------+
|US Airways|         negative|           44|@USAirways 5 hr flight delay and a delay when we land . Is that eve...|
|US Airways|         negative|           32|@USAirways of course never again tho . Thanks for tweetin ur concer...|
|     Delta|         negative|           31|STOP. USING.THIS.WORD. IF. YOU'RE. A. COMPANY. RT @JetBlue: Our fle...|
|US Airways|          neutral|           28|   @USAirways with this livery back in the day. http://t.co/EEqWVAMmiy|
| Southwest|         positive|           22|        @SouthwestAir beautiful day in Seattle! http://t.co/iqu0PPVq2S|
|     Delta|         negative|     